# Graph Fraud Detection — Model Training (ML Job + Tuner API)

Demonstrates two Snowflake-native training paths on the graph-fraud training dataset:

| Step | Method | Description |
|------|--------|-------------|
| 1 | **ML Job** (`submit_directory`) | Baseline XGBoost — runs `source/train.py` on SPCS compute pool |
| 2 | **Tuner API** (BayesOpt) | Distributed HPO in Container Runtime or via `@remote` ML Job |
| 3 | **Model Registry** | Register best model as `FRAUD_RING_MODEL` V1 (prod) and V2 (candidate) |

Both paths use the point-in-time correct training data from `FRAUD_TRAINING_DATA`
(generated in `01_features.ipynb` via `generate_training_set`).

**Run in Container Runtime** for Tuner API support.

In [1]:
import os
import pickle
import time
from datetime import datetime

import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model import type_hints

try:
    from snowflake.ml.modeling.tune import Tuner
    HAS_TUNER = True
except ImportError:
    HAS_TUNER = False

CONNECTION_NAME  = 'parker_demo'
DATABASE         = 'GRAPH_FRAUD_DEMO'
SCHEMA           = 'MODELS'
WAREHOUSE        = 'GRAPH_FRAUD_WH'
COMPUTE_POOL     = 'ML_TRAINING_POOL'
MODEL_NAME       = 'FRAUD_RING_MODEL'
EXPERIMENT_NAME  = 'GRAPH_FRAUD_EXPERIMENT'
TRAINING_TABLE   = f'{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA'
MODEL_STAGE      = f'@{DATABASE}.{SCHEMA}.ML_MODELS'

FEATURE_COLS = [
    'EFFECTIVE_COMMUNITY_SIZE',
    'DEGREE',
    'SHARED_ADDRESS_COUNT',
    'SHARED_IP_COUNT',
    'AVG_COMMUNITY_REFUND',
    'COMMUNITY_REFUND_STDDEV',
    'ADDR_N_FILERS',
    'ADDR_AVG_REFUND',
    'IP_N_FILERS',
    'IP_AVG_REFUND',
    'SUBMISSION_HOUR',
    'IS_HIGH_CLAIM',
    'IS_ODD_HOUR',
    'CLAIM_AMOUNT',
    'HAS_DIRECT_GRAPH',
]

print(f'Tuner API available: {HAS_TUNER}')

# Load PAT from Snowflake config
import tomllib, pathlib
#_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')

In [2]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name=CONNECTION_NAME)
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Connected: {session.get_current_account()}')

## Step 1: Baseline Training via ML Job

Submits `source/train.py` as a job on an SPCS compute pool using
`submit_directory()`. The job runs in isolation — the entire `source/`
directory is uploaded to a stage and executed on SPCS compute.

The 15 graph-community features capture:
- **Direct Leiden features** (community size, degree, shared-attribute counts)
- **Address-inherited** context (applicants at the same address)
- **IP-inherited** context (applicants sharing the same IP)
- **Velocity** signals (filing hour, high claim, odd hour)
- **Request-time** features (claim amount, has-direct-graph flag)

In [3]:
session.sql("DROP GATEWAY IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_GATEWAY").collect()
session.sql("DROP SERVICE IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_SVC_V1").collect()
session.sql("DROP SERVICE IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_SVC_V2").collect()
print("Services + gateway dropped")

session.sql("DROP MODEL IF EXISTS GRAPH_FRAUD_DEMO.MODELS.FRAUD_RING_MODEL").collect()
print("Model dropped")

In [4]:
from snowflake.ml.jobs import submit_directory

session.sql(f'''
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL}
  MIN_NODES = 1 MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
''').collect()
print(f'Compute pool: {COMPUTE_POOL}')

job = submit_directory(
    dir_path='../source',
    entrypoint='train.py',
    compute_pool=COMPUTE_POOL,
    stage_name=f'{DATABASE}.{SCHEMA}.ML_JOB_STAGE',
    env_vars={
        'DATABASE':        DATABASE,
        'SCHEMA':          SCHEMA,
        'EXPERIMENT_NAME': EXPERIMENT_NAME,
        'REGISTER_MODEL':  'true',
        'VERSION_NAME':    'V1',
        'MODEL_STAGE':     MODEL_STAGE,
    },
)
print(f'Job submitted: {job.id}')
print(f'Status      : {job.status}')

In [5]:
print('Waiting for ML Job to complete...')
for i in range(60):
    status = job.status
    print(f'  [{i:02d}] {status}')
    if status in ('DONE', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)

if job.status == 'DONE':
    print(f'\nJob completed successfully')
    try:
        print(job.get_logs())
    except Exception as e:
        print(f'Logs: {e}')
else:
    print(f'\nJob ended with status: {job.status}')
    print(job.get_logs())

### Register Baseline Model as V1 (Production)

Download the trained model artifact from stage, register to Model Registry as `V1`.
This becomes the **production model** the Gateway initially routes 100% traffic to.

In [6]:
# Verify model was registered by the ML Job (no local loading needed)
reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
mv = model_ref.version('V1')

print(f'Model: {MODEL_NAME}')
print(f'Version: V1')
print(f'Metrics: {mv.show_metrics()}')
print(f'Comment: {mv.comment}')
print(f'\nV1 registered successfully — no local artifacts needed.')

## Step 2: Hyperparameter Optimization via Tuner API

Uses **Bayesian Optimization** (UCB acquisition function) to search for the best
XGBoost hyperparameters — 10 trials on a 5-parameter search space.

The Tuner runs in **Container Runtime** if available (in-process on SPCS GPU/CPU),
otherwise falls back to `submit_directory` with manually tuned params.

In [7]:
SEARCH_SPACE = {
    'n_estimators':     (100, 500),
    'max_depth':        (3, 10),
    'learning_rate':    (0.01, 0.3),
    'scale_pos_weight': (5, 30),
    'min_child_weight': (1, 20),
    'gamma':            (0, 5),
    'reg_alpha':        (0, 10),
    'reg_lambda':       (1, 10),
}
NUM_TRIALS = 15

print('BayesOpt search space:')
for k, v in SEARCH_SPACE.items():
    print(f'  {k}: {v}')

In [8]:
def run_hpo(session, search_space_bounds, num_trials):
    """Run distributed HPO — generates training data from Feature Store."""
    from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context
    from snowflake.ml.modeling.tune.search import BayesOpt
    from snowflake.ml.modeling import tune
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.ml.feature_store import FeatureStore, CreationMode
    import xgboost as xgb
    import numpy as np
    from sklearn.metrics import roc_auc_score, f1_score

    session.use_database(DATABASE)
    fs = FeatureStore(
        session=session,
        database=DATABASE,
        name='FEATURES',
        default_warehouse=WAREHOUSE,
        creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
    )

    graph_fv = fs.get_feature_view('GRAPH_COMMUNITY_FEATURES', 'V1')
    try:
        addr_fv = fs.get_feature_view('ADDRESS_COMMUNITY_FEATURES', 'V1')
        ip_fv = fs.get_feature_view('IP_COMMUNITY_FEATURES', 'V1')
        extra_fvs = [addr_fv, ip_fv]
        print('  Including ADDRESS + IP FVs in training set')
    except Exception:
        extra_fvs = []
        print('  ADDRESS/IP FVs not found — training on graph FV only')

    spine_df = session.sql(f"""
        SELECT
            cr.APPLICANT_ID::VARCHAR AS APPLICANT_ID,
            tr.ADDRESS_ID::VARCHAR AS ADDRESS_ID,
            tr.IP_ADDRESS::VARCHAR AS IP_ADDRESS,
            tr.SUBMITTED_AT,
            tr.CLAIM_AMOUNT::DOUBLE AS CLAIM_AMOUNT,
            HOUR(tr.SUBMITTED_AT)::DOUBLE AS SUBMISSION_HOUR,
            CASE WHEN tr.CLAIM_AMOUNT > 10000 THEN 1.0 ELSE 0.0 END AS IS_HIGH_CLAIM,
            CASE WHEN HOUR(tr.SUBMITTED_AT) BETWEEN 0 AND 5 THEN 1.0 ELSE 0.0 END AS IS_ODD_HOUR,
            CASE WHEN cr.COMMUNITY_SIZE > 0 THEN cr.COMMUNITY_SIZE ELSE 0.0 END::DOUBLE AS EFFECTIVE_COMMUNITY_SIZE,
            CASE WHEN cr.COMMUNITY_SIZE > 0 THEN 1.0 ELSE 0.0 END AS HAS_DIRECT_GRAPH,
            CASE WHEN frm.APPLICANT_ID IS NOT NULL THEN 1.0 ELSE 0.0 END AS IS_FRAUD
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS cr
        JOIN {DATABASE}.RAW.APPLICATIONS tr ON tr.APPLICANT_ID = cr.APPLICANT_ID
        LEFT JOIN {DATABASE}.RAW.FRAUD_RING_MEMBERS frm ON frm.APPLICANT_ID = cr.APPLICANT_ID
        QUALIFY ROW_NUMBER() OVER (PARTITION BY cr.APPLICANT_ID ORDER BY tr.SUBMITTED_AT DESC) = 1
    """)

    fraud_df = fs.generate_training_set(
        spine_df=spine_df,
        features=[graph_fv] + extra_fvs,
        spine_timestamp_col='SUBMITTED_AT',
        spine_label_cols=['IS_FRAUD'],
    )
    print(f'  Generated training set from Feature Store')

    train_df = fraud_df.sample(frac=0.8)
    test_df  = fraud_df.subtract(train_df)

    dataset_map = {
        'train': DataConnector.from_dataframe(train_df),
        'test':  DataConnector.from_dataframe(test_df),
    }

    search_space = {k: tune.uniform(v[0], v[1]) for k, v in search_space_bounds.items()}

    def train_fn():
        ctx = get_tuner_context()
        config = ctx.get_hyper_params()
        dm = ctx.get_dataset_map()
        train_pd = dm['train'].to_pandas()
        test_pd  = dm['test'].to_pandas()

        available = [c for c in FEATURE_COLS if c in train_pd.columns]
        X_train = train_pd[available].fillna(0)
        y_train = train_pd['IS_FRAUD'].fillna(0)
        X_test  = test_pd[available].fillna(0)
        y_test  = test_pd['IS_FRAUD'].fillna(0)

        m = xgb.XGBClassifier(
            n_estimators=int(config['n_estimators']),
            max_depth=int(config['max_depth']),
            learning_rate=config['learning_rate'],
            scale_pos_weight=config['scale_pos_weight'],
            min_child_weight=config['min_child_weight'],
            gamma=config.get('gamma', 0),
            reg_alpha=config.get('reg_alpha', 0),
            reg_lambda=config.get('reg_lambda', 1),
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss',
        )
        m.fit(X_train, y_train)
        y_proba = m.predict_proba(X_test)[:, 1]

        # Threshold-optimized F1 as the tuner metric
        thresholds = np.arange(0.1, 0.9, 0.01)
        best_f1 = 0.0
        for t in thresholds:
            f1 = f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1 = f1

        auc = roc_auc_score(y_test, y_proba)
        ctx.report(metrics={'f1': best_f1, 'auc': auc}, model=m)

    tuner_config = TunerConfig(
        metric='f1',
        mode='max',
        search_alg=BayesOpt(utility_kwargs={'kind': 'ucb', 'kappa': 2.5}),
        num_trials=num_trials,
        uses_snowflake_trainer=False,
        max_concurrent_trials=1,
    )

    tuner = Tuner(
        train_func=train_fn,
        search_space=search_space,
        tuner_config=tuner_config,
    )

    return tuner.run(dataset_map=dataset_map)

In [9]:
if HAS_TUNER:
    print(f'Running {NUM_TRIALS}-trial BayesOpt HPO in Container Runtime...')
    hpo_results = run_hpo(session, SEARCH_SPACE, NUM_TRIALS)
else:
    print('Tuner API not available locally — submitting as @remote ML Job...')
    from snowflake.ml.jobs import submit_directory
    hpo_job = submit_directory(
        dir_path='../source',
        entrypoint='train.py',
        compute_pool=COMPUTE_POOL,
        stage_name=f'{DATABASE}.{SCHEMA}.ML_JOB_STAGE',
        env_vars={
            'DATABASE':        DATABASE,
            'SCHEMA':          SCHEMA,
            'EXPERIMENT_NAME': EXPERIMENT_NAME,
            'MODEL_STAGE':     MODEL_STAGE,
            'REGISTER_MODEL':  'true',
            'VERSION_NAME':    'V2',
            'N_ESTIMATORS':    '300',
            'MAX_DEPTH':       '6',
            'LEARNING_RATE':   '0.03',
        },
    )
    print(f'HPO job submitted: {hpo_job.id}')
    for i in range(60):
        s = hpo_job.status
        print(f'  [{i:02d}] {s}')
        if s in ('DONE', 'FAILED', 'CANCELLED'):
            break
        time.sleep(30)
    print(hpo_job.get_logs())

### Extract Best Params & Register as V2 (Candidate)

In [10]:
if HAS_TUNER:
    import numpy as np
    from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
    from sklearn.model_selection import train_test_split
    from snowflake.ml.feature_store import FeatureStore, CreationMode

    best_row    = hpo_results.best_result.iloc[0]
    best_params = {
        'n_estimators':     int(best_row['config/n_estimators']),
        'max_depth':        int(best_row['config/max_depth']),
        'learning_rate':    float(best_row['config/learning_rate']),
        'scale_pos_weight': float(best_row['config/scale_pos_weight']),
        'min_child_weight': float(best_row['config/min_child_weight']),
    }
    for p in ['gamma', 'reg_alpha', 'reg_lambda']:
        key = f'config/{p}'
        if key in best_row.index:
            best_params[p] = float(best_row[key])
    best_f1 = float(best_row.get('f1', best_row.get('auc', 0)))
    print(f'Best params : {best_params}')
    print(f'Best F1     : {best_f1:.4f}')

    tuned_model = hpo_results.best_model

    run_name = f'xgb_graph_fraud_tuned_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    exp = ExperimentTracking(session=session, database_name=DATABASE, schema_name=SCHEMA)
    exp.set_experiment(EXPERIMENT_NAME)

    # Generate evaluation data from Feature Store
    session.use_database(DATABASE)
    fs = FeatureStore(
        session=session,
        database=DATABASE,
        name='FEATURES',
        default_warehouse=WAREHOUSE,
        creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
    )
    graph_fv = fs.get_feature_view('GRAPH_COMMUNITY_FEATURES', 'V1')
    try:
        addr_fv = fs.get_feature_view('ADDRESS_COMMUNITY_FEATURES', 'V1')
        ip_fv = fs.get_feature_view('IP_COMMUNITY_FEATURES', 'V1')
        extra_fvs = [addr_fv, ip_fv]
    except Exception:
        extra_fvs = []

    spine_df = session.sql(f"""
        SELECT
            cr.APPLICANT_ID::VARCHAR AS APPLICANT_ID,
            tr.ADDRESS_ID::VARCHAR AS ADDRESS_ID,
            tr.IP_ADDRESS::VARCHAR AS IP_ADDRESS,
            tr.SUBMITTED_AT,
            tr.CLAIM_AMOUNT::DOUBLE AS CLAIM_AMOUNT,
            HOUR(tr.SUBMITTED_AT)::DOUBLE AS SUBMISSION_HOUR,
            CASE WHEN tr.CLAIM_AMOUNT > 10000 THEN 1.0 ELSE 0.0 END AS IS_HIGH_CLAIM,
            CASE WHEN HOUR(tr.SUBMITTED_AT) BETWEEN 0 AND 5 THEN 1.0 ELSE 0.0 END AS IS_ODD_HOUR,
            CASE WHEN cr.COMMUNITY_SIZE > 0 THEN cr.COMMUNITY_SIZE ELSE 0.0 END::DOUBLE AS EFFECTIVE_COMMUNITY_SIZE,
            CASE WHEN cr.COMMUNITY_SIZE > 0 THEN 1.0 ELSE 0.0 END AS HAS_DIRECT_GRAPH,
            CASE WHEN frm.APPLICANT_ID IS NOT NULL THEN 1.0 ELSE 0.0 END AS IS_FRAUD
        FROM {DATABASE}.GRAPH.COMMUNITY_RESULTS cr
        JOIN {DATABASE}.RAW.APPLICATIONS tr ON tr.APPLICANT_ID = cr.APPLICANT_ID
        LEFT JOIN {DATABASE}.RAW.FRAUD_RING_MEMBERS frm ON frm.APPLICANT_ID = cr.APPLICANT_ID
        QUALIFY ROW_NUMBER() OVER (PARTITION BY cr.APPLICANT_ID ORDER BY tr.SUBMITTED_AT DESC) = 1
    """)

    train_pdf = fs.generate_training_set(
        spine_df=spine_df,
        features=[graph_fv] + extra_fvs,
        spine_timestamp_col='SUBMITTED_AT',
        spine_label_cols=['IS_FRAUD'],
    ).to_pandas()

    model_features = tuned_model.get_booster().feature_names
    X = train_pdf[model_features].fillna(0)
    y = train_pdf['IS_FRAUD'].fillna(0)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    y_proba = tuned_model.predict_proba(X_test)[:, 1]

    # Threshold optimization: sweep to maximize F1
    thresholds = np.arange(0.05, 0.95, 0.01)
    f1_scores = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_threshold = thresholds[np.argmax(f1_scores)]
    print(f'Optimal threshold: {best_threshold:.2f} (F1: {max(f1_scores):.4f} vs default-0.5 F1: {f1_score(y_test, (y_proba >= 0.5).astype(int), zero_division=0):.4f})')

    y_pred = (y_proba >= best_threshold).astype(int)
    tuned_metrics = {
        'auc':       float(roc_auc_score(y_test, y_proba)),
        'recall':    float(recall_score(y_test, y_pred, zero_division=0)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'f1':        float(f1_score(y_test, y_pred, zero_division=0)),
        'threshold': float(best_threshold),
    }
    print(f'Test metrics (threshold={best_threshold:.2f}): {tuned_metrics}')

    with exp.start_run(run_name):
        exp.log_params({**{k: str(round(v, 4)) for k, v in best_params.items()}, 'run_type': 'graph_fraud_tuner', 'threshold': str(round(best_threshold, 4))})
        exp.log_metrics({'test_auc_roc': tuned_metrics['auc'], 'test_f1': tuned_metrics['f1'], 'test_precision': tuned_metrics['precision'], 'test_recall': tuned_metrics['recall']})
    print(f'Logged run: {run_name}')

    sample_df = session.create_dataframe(X_test.head(100))
    reg.log_model(
        model=tuned_model,
        model_name=MODEL_NAME,
        version_name='V2',
        sample_input_data=sample_df,
        metrics=tuned_metrics,
        comment=f'Tuned graph fraud model (BayesOpt) — F1: {tuned_metrics["f1"]:.4f}, threshold: {best_threshold:.2f}',
        target_platforms=[type_hints.TargetPlatform.SNOWPARK_CONTAINER_SERVICES],
    )
    print(f'Registered: {MODEL_NAME}:V2 (candidate — F1 {tuned_metrics["f1"]:.4f})')

## Registry Summary

In [11]:
model_ref = reg.get_model(MODEL_NAME)
print(f'Model: {MODEL_NAME}')
for mv in model_ref.versions():
    m = mv.show_metrics()
    auc = m.get('auc', None)
    auc_str = f'{auc:.4f}' if isinstance(auc, (int, float)) else 'N/A'
    print(f'  {mv.version_name}  AUC={auc_str}  comment="{mv.comment}"')
print('\nProceed to 03_deploy_gateway.ipynb')